In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip /content/drive/MyDrive/pgl_hogye_final_hohi_nii_rha_returns.zip -d /content/dataset2/

Streaming output truncated to the last 5000 lines.
  inflating: /content/dataset2/pgl_hogye_final_hohi_nii _rha/artifacts/Texture bleeding between adjacent regions/perturbed_4553_aug_1.jpg  
  inflating: /content/dataset2/pgl_hogye_final_hohi_nii _rha/artifacts/Texture bleeding between adjacent regions/perturbed_4553_aug_2.jpg  
  inflating: /content/dataset2/pgl_hogye_final_hohi_nii _rha/artifacts/Texture bleeding between adjacent regions/perturbed_4553_aug_3.jpg  
  inflating: /content/dataset2/pgl_hogye_final_hohi_nii _rha/artifacts/Texture bleeding between adjacent regions/perturbed_4553_aug_4.jpg  
  inflating: /content/dataset2/pgl_hogye_final_hohi_nii _rha/artifacts/Texture bleeding between adjacent regions/perturbed_4541_aug_0.jpg  
  inflating: /content/dataset2/pgl_hogye_final_hohi_nii _rha/artifacts/Texture bleeding between adjacent regions/perturbed_4541_aug_1.jpg  
  inflating: /content/dataset2/pgl_hogye_final_hohi_nii _rha/artifacts/Texture bleeding between adjacent regi

In [ ]:
import os
import random
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import ViTModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import joblib  # For saving the trained models

# Paths to your data directories
data_dir = '/content/dataset2/pgl_hogye_final_hohi_nii _rha'
artifacts_dir = os.path.join(data_dir, 'artifacts')

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Transformation for input images
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    # Normalization parameters for ViT
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5]),
])

# Custom Dataset class
class ImageDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

# Parameters
batch_size = 32

# List of artifact directories
artifact_list = sorted(os.listdir(artifacts_dir))

# Pre-load all artifact images
artifact_images_dict = {}
for artifact_name in artifact_list:
    artifact_folder = os.path.join(artifacts_dir, artifact_name)
    images = [os.path.join(artifact_folder, img) for img in os.listdir(artifact_folder)]
    artifact_images_dict[artifact_name] = images

# Load pre-trained ViT model for feature extraction (load only once)
print("Loading ViT model...")
vit_model = ViTModel.from_pretrained('google/vit-large-patch16-224-in21k')
vit_model.to(device)
vit_model.eval()  # Set model to evaluation mode

# Function to extract embeddings
def extract_embeddings(data_loader):
    embeddings = []
    labels_list = []
    with torch.no_grad():
        for images_batch, labels_batch in data_loader:
            images_batch = images_batch.to(device)
            outputs = vit_model(images_batch)
            cls_embeddings = outputs.last_hidden_state[:, 0, :]  # [CLS] token
            embeddings.append(cls_embeddings.cpu().numpy())
            labels_list.extend(labels_batch.numpy())
    embeddings = np.vstack(embeddings)
    labels_list = np.array(labels_list)
    return embeddings, labels_list

# Loop over each artifact
for artifact_num in range(len(artifact_list)):
    artifact_name = artifact_list[artifact_num]
    print(artifact_num, f"\nProcessing artifact: {artifact_name}")

    # Get images with the artifact (positive samples)
    artifact_images = artifact_images_dict[artifact_name]
    num_artifact_images = len(artifact_images)

    # Get images from other artifacts (negative samples)
    other_artifact_images = []
    for other_name, images in artifact_images_dict.items():
        if other_name != artifact_name:
            other_artifact_images.extend(images)

    # Ensure there are enough images
    if num_artifact_images > len(other_artifact_images):
        raise ValueError(f"Not enough images in other artifacts for artifact '{artifact_name}'.")

    # Randomly select negative samples to balance the dataset
    negative_images = random.sample(other_artifact_images, num_artifact_images)

    # Combine and label the data
    images = artifact_images + negative_images
    labels = [1] * num_artifact_images + [0] * num_artifact_images

    # Shuffle the dataset
    combined = list(zip(images, labels))
    random.shuffle(combined)
    images[:], labels[:] = zip(*combined)

    # Split into training and validation sets (80% train, 20% validation)
    train_images, val_images, train_labels, val_labels = train_test_split(
        images, labels, test_size=0.2, random_state=42, stratify=labels)

    # Create datasets
    train_dataset = ImageDataset(train_images, train_labels, transform=transform)
    val_dataset = ImageDataset(val_images, val_labels, transform=transform)

    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # Extract embeddings for training and validation sets
    print("Extracting embeddings for training set...")
    train_embeddings, train_labels_array = extract_embeddings(train_loader)

    print("Extracting embeddings for validation set...")
    val_embeddings, val_labels_array = extract_embeddings(val_loader)

    # Train RandomForestClassifier
    print("Training RandomForestClassifier...")
    rf_classifier = RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features='sqrt',
        bootstrap=True,
        random_state=42
    )
    rf_classifier.fit(train_embeddings, train_labels_array)

    # Evaluate on validation set
    val_predictions = rf_classifier.predict(val_embeddings)
    val_accuracy = accuracy_score(val_labels_array, val_predictions)
    print(f'Artifact: {artifact_name}, Validation Accuracy: {val_accuracy * 100:.2f}%')

    # Save the trained RandomForestClassifier model in binary format
    model_save_path = f'/content/drive/MyDrive/rf_models/rf_model_{artifact_name}.pth'  # Using .pth extension
    joblib.dump(rf_classifier, model_save_path)
    print(f"Model saved to {model_save_path}")


Loading ViT model...
0 
Processing artifact: Abruptly cut-off objects
Extracting embeddings for training set...
Extracting embeddings for validation set...
Training RandomForestClassifier...
Artifact: Abruptly cut-off objects, Validation Accuracy: 92.33%
Model saved to /content/drive/MyDrive/rf_models/rf_model_Abruptly cut-off objects.pth
1 
Processing artifact: Aliasing along high-contrast edges
Extracting embeddings for training set...
Extracting embeddings for validation set...
Training RandomForestClassifier...
Artifact: Aliasing along high-contrast edges, Validation Accuracy: 89.17%
Model saved to /content/drive/MyDrive/rf_models/rf_model_Aliasing along high-contrast edges.pth
2 
Processing artifact: Anatomically_Impossible_Joint_Configurations
Extracting embeddings for training set...
Extracting embeddings for validation set...
Training RandomForestClassifier...
Artifact: Anatomically_Impossible_Joint_Configurations, Validation Accuracy: 91.67%
Model saved to /content/drive/MyDri